In [ ]:
#Loading and computing class counts:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv("../data/raw/isic/ISIC_2019_Training_GroundTruth.csv")
class_cols = [c for c in df.columns if c != "image"]
df["label"] = df[class_cols].idxmax(axis=1)

class_counts = df["label"].value_counts()
print(class_counts)

label
NV      12875
MEL      4522
BCC      3323
BKL      2624
AK        867
SCC       628
VASC      253
DF        239
Name: count, dtype: int64


In [2]:
#Define floor (rare classes, take all) and cap (dominant classes):
RARE_THRESHOLD = 200    
CAP            = 2500   
TARGET_TOTAL   = 10000


rare_classes = class_counts[class_counts < RARE_THRESHOLD].index.tolist()
dominant_classes = class_counts[class_counts > 1000].index.tolist()
mid_classes = [c for c in class_counts.index if c not in rare_classes and c not in dominant_classes]

print("Rare:", rare_classes)
print("Dominant:", dominant_classes)
print("Mid:", mid_classes)

Rare: []
Dominant: ['NV', 'MEL', 'BCC', 'BKL']
Mid: ['AK', 'SCC', 'VASC', 'DF']


In [3]:
# Sample according to floor/cap/proportional logic:
sampled_parts = []

# rare classes: take everything
for cls in rare_classes:
    sampled_parts.append(df[df["label"] == cls])

# dominant classes: cap
for cls in dominant_classes:
    cls_df = df[df["label"] == cls]
    sampled_parts.append(cls_df.sample(n=min(CAP, len(cls_df)), random_state=42))

rare_dominant_count = sum(len(p) for p in sampled_parts)
remaining_budget = TARGET_TOTAL - rare_dominant_count
per_mid_class = remaining_budget // max(len(mid_classes), 1)

for cls in mid_classes:
    cls_df = df[df["label"] == cls]
    sampled_parts.append(cls_df.sample(n=min(per_mid_class, len(cls_df)), random_state=42))

isic_subset = pd.concat(sampled_parts).reset_index(drop=True)
print(isic_subset.shape)
print(isic_subset["label"].value_counts())

(10000, 11)
label
NV     2500
MEL    2500
BCC    2500
BKL    2500
Name: count, dtype: int64


In [ ]:
#Building standardised manifest and saveing:
manifest = pd.DataFrame({
    "image_id": isic_subset["image"],
    "image_path": isic_subset.apply(lambda row: f"{row['label']}/{row['image']}.jpg", axis=1),
    "label": isic_subset["label"],
    "is_uncertain_original": False,
    "patient_id": isic_subset["image"],  # synthetic, no patient grouping in ISIC
})

manifest.to_csv("../data/subsets/isic_subset_manifest.csv", index=False)
print("Saved:", manifest.shape)

Saved: (10000, 5)
